In [1]:
import torch
from torch import nn
from torch.nn import functional as F
net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
x = torch.randn(2, 20)
net(x)


tensor([[-0.0033, -0.0010, -0.2294,  0.4809, -0.2675, -0.3877, -0.2775,  0.0464,
          0.0380,  0.1035],
        [ 0.0154,  0.1888, -0.1455,  0.2307, -0.1863, -0.5464, -0.2102,  0.4331,
         -0.1250,  0.3043]], grad_fn=<AddmmBackward0>)

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()  #调用MLP父类的构造函数进行初始化
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)
    def forward(self, x):   #定义模型的前向传播，即如何根据输入x计算输出
        return self.out(F.relu(self.hidden(x)))

In [4]:
net  = MLP()
net(x)

tensor([[-0.2882,  0.1583, -0.7487,  0.2314,  0.3218, -0.0952,  0.6383, -0.1471,
         -0.4961, -0.1262],
        [ 0.0500,  0.0745, -0.0800, -0.1582, -0.0837, -0.0930,  0.2886, -0.3940,
         -0.3859,  0.1142]], grad_fn=<AddmmBackward0>)

In [9]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            self.add_module(str(idx), module)
    def forward(self, x):
        for module in self.children():
            x = module(x)
        return x

In [11]:
net = MySequential(nn.Linear(20, 256),nn.ReLU(), nn.Linear(256, 10))
net(x)

tensor([[ 0.1637, -0.7149,  0.6682,  0.1415,  0.3958, -0.3431, -0.2764, -0.0974,
          0.4954, -0.1374],
        [-0.1107,  0.0473,  0.8498, -0.2705,  0.1112, -0.0956, -0.1534, -0.4305,
          0.0791,  0.3487]], grad_fn=<AddmmBackward0>)

In [14]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.rand_weight = torch.randn((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)
    def forward(self, x):
        x = self.linear(x)
        x = F.relu(torch.matmul(x, self.rand_weight))
        x = self.linear(x)
        while x.abs().sum() > 1:
            x /= 2
        return x.sum()

In [16]:
net = FixedHiddenMLP()
net(x)

tensor(-0.0108, grad_fn=<SumBackward0>)

In [18]:
class NestMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(),
                                 nn.Linear(64, 32), nn.ReLU())
        self.linear = nn.Linear(32, 16)

    def forward(self, x):
        return self.linear(self.net(x))

chimera = nn.Sequential(NestMLP(), nn.Linear(16, 20), FixedHiddenMLP())
chimera(x)

tensor(0.1232, grad_fn=<SumBackward0>)